# 📘 Intelligent Mentoring System Boilerplate
## Using Mehyaar/Annotated_NER_PDF_Resumes Dataset

In [ ]:
!pip install resume-parser --quiet

In [ ]:
# Test the modified resume-parser
import json
import os

print("Testing modified resume-parser (bypassing custom models)...")

try:
    from resume_parser import resumeparse
    print("✅ Resume-parser imported successfully!")
    
    # Load the dataset
    directory_path = "data/ResumesJsonAnnotated"
    
    # Load all JSON files
    data = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".json"):
            with open(os.path.join(directory_path, filename), "r") as file:
                data.append(json.load(file))
    
    print(f"✅ Loaded {len(data)} resumes from the dataset")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Test resume-parser with fixed patterns
import importlib
import sys

# Force reload of resume_parser module to get latest fixes
if 'resume_parser' in sys.modules:
    del sys.modules['resume_parser']
if 'resume_parser.resumeparse' in sys.modules:
    del sys.modules['resume_parser.resumeparse']

import tempfile
import os

# Fresh import after fixes
from resume_parser import resumeparse

# Test resume-parser on first resume
sample_resume = data[0]
print("Testing fixed resume-parser...")
print("Sample resume text (first 300 chars):")
print(sample_resume['text'][:300])
print("\n" + "="*50)

# Create a temporary file with the resume text
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as temp_file:
    temp_file.write(sample_resume['text'])
    temp_file_path = temp_file.name

try:
    # Parse the resume using resume-parser
    parsed_data = resumeparse.read_file(temp_file_path)
    
    print("✅ SUCCESS! Parsed data from resume-parser:")
    print("="*50)
    for key, value in parsed_data.items():
        if value:  # Only show non-empty fields
            if isinstance(value, list) and len(value) > 3:
                print(f"{key}: {value[:3]}... ({len(value)} total)")
            else:
                print(f"{key}: {value}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # Clean up the temporary file
    os.unlink(temp_file_path)

In [ ]:
# Parse all resumes with robust Unicode and error handling
import tempfile
import os
import logging

# Suppress resume-parser logging errors
logging.getLogger().setLevel(logging.CRITICAL)

print("Processing all resumes with robust Unicode and error handling...")
print("="*50)

successful_parses = 0
failed_parses = 0
error_types = {}

def clean_unicode_text(text):
    """Clean text to handle Unicode surrogates and other issues."""
    if not isinstance(text, str):
        text = str(text)
    
    # Handle Unicode surrogates and other problematic characters
    cleaned = text.encode('utf-8', errors='ignore').decode('utf-8')
    
    # Additional cleaning for common problematic characters
    cleaned = cleaned.replace('\ud83d', '')  # Remove emoji surrogates
    cleaned = cleaned.replace('\udcxx', '')  # Remove other surrogates
    
    return cleaned

def safe_parse_resume(text, idx):
    """Safely parse a resume with comprehensive error handling."""
    temp_file_path = None
    try:
        # Clean text first
        clean_text = clean_unicode_text(text)
        
        # Create temporary file with error handling
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8', errors='ignore') as temp_file:
            temp_file.write(clean_text)
            temp_file_path = temp_file.name
        
        # Parse the resume
        parsed = resumeparse.read_file(temp_file_path)
        
        # Validate parsed data
        if not isinstance(parsed, dict):
            raise ValueError("Parser returned invalid data type")
        
        # Clean up any None or problematic values
        cleaned_parsed = {}
        for key, value in parsed.items():
            if value is not None:
                if isinstance(value, list):
                    # Filter out None values and clean strings in lists
                    cleaned_list = []
                    for v in value:
                        if v is not None:
                            if isinstance(v, str):
                                cleaned_list.append(clean_unicode_text(v))
                            else:
                                cleaned_list.append(v)
                    cleaned_parsed[key] = cleaned_list
                elif isinstance(value, str):
                    cleaned_parsed[key] = clean_unicode_text(value)
                else:
                    cleaned_parsed[key] = value
        
        return cleaned_parsed
        
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)[:50]  # First 50 chars of error
        error_key = f"{error_type}: {error_msg}"
        error_types[error_key] = error_types.get(error_key, 0) + 1
        return None
        
    finally:
        # Always clean up temp file
        if temp_file_path and os.path.exists(temp_file_path):
            try:
                os.unlink(temp_file_path)
            except:
                pass

# Process all resumes
print("Starting processing...")
for idx, resume_data in enumerate(data):
    parsed = safe_parse_resume(resume_data['text'], idx)
    
    if parsed is not None:
        resume_data['parsed'] = parsed
        successful_parses += 1
    else:
        resume_data['parsed'] = None
        failed_parses += 1
    
    # Show progress every 100 resumes
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(data)} resumes... ✅ Success: {successful_parses}, ❌ Failed: {failed_parses}")

print(f"\n{'='*60}")
print(f"FINAL RESULTS:")
print(f"{'='*60}")
print(f"✅ Successfully parsed: {successful_parses}")
print(f"❌ Failed to parse: {failed_parses}")
print(f"📊 Success rate: {(successful_parses/len(data)*100):.1f}%")

if error_types:
    print(f"\n📋 Top error types:")
    for error_key, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {error_key}: {count} occurrences")

# Reset logging level
logging.getLogger().setLevel(logging.WARNING)

print(f"\n🎉 Processing complete! Ready to save data.")

In [ ]:
# Save the data variable to file (with Unicode handling)
import json

# Clean data to handle Unicode issues
def clean_unicode(obj):
    if isinstance(obj, str):
        # Replace problematic Unicode characters
        return obj.encode('utf-8', errors='ignore').decode('utf-8')
    elif isinstance(obj, dict):
        return {k: clean_unicode(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_unicode(item) for item in obj]
    else:
        return obj

print("Cleaning Unicode characters...")
cleaned_data = clean_unicode(data)

# Save the cleaned data
with open('parsed_data.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=True)

print("✅ Data saved to 'parsed_data.json'")
print(f"📊 Saved {len(cleaned_data)} resumes")

# To load later:
# with open('parsed_data.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

In [ ]:
# Save parsed data to avoid reprocessing
import json
import os
from datetime import datetime

# Create output directory if it doesn't exist
output_dir = "parsed_data"
os.makedirs(output_dir, exist_ok=True)

# Save the parsed data with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_dir, f"parsed_resumes_{timestamp}.json")

print(f"Saving parsed data to: {output_file}")
print("="*50)

# Prepare data for saving (only successfully parsed resumes)
parsed_resumes = []
for idx, item in enumerate(data):
    if item.get('parsed'):
        resume_entry = {
            'index': idx,
            'original_text': item['text'],
            'annotations': item.get('annotations', []),
            'parsed_data': item['parsed']
        }
        parsed_resumes.append(resume_entry)

# Save to JSON file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump({
        'metadata': {
            'total_resumes': len(data),
            'successfully_parsed': len(parsed_resumes),
            'success_rate': f"{(len(parsed_resumes)/len(data)*100):.1f}%",
            'processed_date': datetime.now().isoformat(),
            'parser_used': 'resume-parser'
        },
        'resumes': parsed_resumes
    }, f, indent=2, ensure_ascii=False)

print(f"✅ Successfully saved {len(parsed_resumes)} parsed resumes")
print(f"📊 Success rate: {(len(parsed_resumes)/len(data)*100):.1f}%")
print(f"📁 File size: {os.path.getsize(output_file)/1024/1024:.2f} MB")

# Also save a summary file with just statistics
summary_file = os.path.join(output_dir, f"parsing_summary_{timestamp}.json")
summary_stats = {
    'total_resumes': len(data),
    'successfully_parsed': len(parsed_resumes),
    'failed_parses': len(data) - len(parsed_resumes),
    'success_rate': f"{(len(parsed_resumes)/len(data)*100):.1f}%",
    'available_fields': list(sample_parsed.keys()) if successful_parses > 0 else [],
    'field_coverage': {}
}

# Calculate field coverage
if len(parsed_resumes) > 0:
    for field in summary_stats['available_fields']:
        count = sum(1 for resume in parsed_resumes if resume['parsed_data'].get(field))
        coverage = (count / len(parsed_resumes)) * 100
        summary_stats['field_coverage'][field] = f"{coverage:.1f}%"

with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary_stats, f, indent=2)

print(f"✅ Summary saved to: {summary_file}")
print("\n📋 Field Coverage:")
for field, coverage in summary_stats['field_coverage'].items():
    print(f"  {field}: {coverage}")

print(f"\n💡 To load the data later, use:")
print(f"   with open('{output_file}', 'r') as f:")
print(f"       loaded_data = json.load(f)")
print(f"   resumes = loaded_data['resumes']")